## Dropout

1. Regularization for neural nets: randomly zeroes out neurons during training only (prevents units from co-adapting too strongly). Every neuron is a weighted sum of inputs followed by activation.
2. Example: 100 inputs, each weight=1, each input=1 → neuron input = 100. At 40% dropout, ~40 of those are zeroed → effective input ≈ 60 during training vs. 100 at inference, a train/inference mismatch that needs correcting.
3. Inverted dropout fixes it: sample a mask $m_i\sim\text{Bernoulli}(1-p)$, apply $x=x*m$, then rescale by $1/(1-p)$. At $p=0.4$: $60\times(1/(1-0.4))=100$, training and inference now see the same expected scale.
4. Why it helps: acts as an implicit ensemble of subnetworks, each training pass uses a different random subnetwork, reducing overfitting by not letting the model rely on any single feature path.
5. Common rates: 0.1–0.3 for large models, 0.4–0.5 for smaller fully-connected nets. Too high a rate underfits.


In [ ]:
import torch
import torch.nn as nn

dropout_layer = nn.Dropout(0.2)
p = 0.2
x = torch.randn((1, 5))

print(x)
print(x/(1-p)) # same as dropout
print(dropout_layer(x)) # dropout layer

# eval mode neither drops nor scales:
dropout_layer.eval()
print(dropout_layer(x))

## Bias-Variance Tradeoff & the Broader Regularization Toolkit

Dropout above is one specific regularization technique; this is the bigger picture it fits into, and the other tools that solve the same bias-variance problem in different ways. See also [XGBoost, from scratch](xgboost-from-scratch.ipynb) for shrinkage (η) as this same tradeoff's boosting-specific knob.

1. Bias = underfitting (an overly simple model, e.g. linear regression on nonlinear data). Variance = overfitting (the model memorizes noise, e.g. an unpruned decision tree, KNN with K=1, or an unregularized neural network). High bias hurts both train and test error equally (the model is too rigid to fit either); high variance shows low train error but high validation/test error (the model memorized the training noise).
2. Explicit penalty terms: L1/Lasso adds λΣ|θᵢ| — drives redundant weights to exactly zero, an implicit feature selector. L2/Ridge adds λΣθᵢ² — penalizes large weights uniformly rather than zeroing them. Elastic Net combines both, handling correlated features better than L1 alone.
3. Structural constraints: tree pruning / max_depth / min_samples_split stop a tree memorizing rare leaves. Dropout (covered above) randomly deactivates neurons per step. Early stopping halts training when validation error starts rising. Data augmentation increases effective sample diversity, forcing invariant representations. Bagging (e.g. Random Forests) trains multiple models on bootstrap samples and averages them, reducing variance roughly by a factor of 1/N.
4. Cross-validation's role: a single train/test split can pick an unreliable model-complexity choice if the holdout happens to be unrepresentative. K-Fold CV trains on K−1 folds and tests on the remaining one, repeating K times and averaging — this dampens single-split noise and makes hyperparameter choices reflect stability across multiple subsets rather than one lucky/unlucky split. Stratified K-Fold additionally preserves class proportions in every fold, avoiding artificial variance from class imbalance in smaller folds.